Intermediate Deep Learning - Deep Reinforcement Learning
# Project: Deep Reinforcement Learning in Trading Environment

## Overview
In this project, you will implement a deep reinforcement learning (DRL) algorithm covered in this course and train an agent in a trading environment. You will then use the observed results to guide and assess improvements to your initial implementation.

### Team Organisation

Work in teams of **exactly two students** (except for one pre-approved individual group). Each team submits one shared code implementation and evaluation CSV file, and gives one presentation.

### Project Requirements

1. **Implement course algorithms.** Choose one or two DRL algorithms covered in the course, implement them, and train an initial agent to establish a baseline. Explain why the chosen algorithm(s) are suitable for the task.
2. **Analyse the initial results.** Examine training behaviour and agent performance to identify limitations and opportunities for improvement.
3. **Optimise based on evidence.** Use your observations to motivate changes, for example to hyperparameters, network architecture, input features, reward design, or the training procedure. Compare the results with your baseline and explain what worked, what did not, and why.

### Deliverables and Schedule

- **Upload your code:** Jupyter Notebooks or Python scripts containing the algorithm implementation, training, optimisation experiments, and evaluation.
- **Upload `evaluation_results.csv`:** Generate this file for your final agent using the provided evaluation function.
- **Give a presentation of no more than 10 minutes per team, followed by Q&A:** Explain your choice of algorithm, the initial results, your optimisation ideas and their rationale, the experiments conducted, and the resulting performance compared with the baseline.

No written report is required. Upload the code and evaluation CSV **before your presentation**. Presentations will take place during **the last class session**.

### Environment
We will use Gym Trading Env as our trading environment. (https://gym-trading-env.readthedocs.io/en/latest/)

- This is a gymnasium-compatible environment designed to simulate trading (stocks or crypto) from historical market data.
- Its goal is to provide a fast and customizable platform for training RL agents in a trading scenario.

We will use BTC/USDT hour step historical data from Binance for training and evaluation. All students must use the same fixed evaluation period: **2026-08-17 00:00 UTC to 2026-09-17 00:00 UTC (exclusive)**, covering all complete days from August 17 through September 16. Do not change the evaluation dates for submission. Training uses only data from before the test period, with no overlap between training and test data.

Following code blocks demonstrate how to set up the environment and evaluate your agent.

### Grading Criteria

- **Implementation — 25%:** Correct implementation of a DRL algorithm covered in the course, with a functioning training and evaluation process.
- **Optimisation — 50%:** Analysis of observed results, well-motivated improvements, systematic experiments, and clear comparisons with the initial baseline. Explain both successful and unsuccessful changes.
- **Presentation — 25%:** Clear explanation of the algorithm choice, optimisation rationale, experimental results, and conclusions.

**Bonus:** The **top 3 teams** ranked by agent performance on the submitted evaluation results will each receive a **10% bonus** on their project grade.


---

## Environment Setup

### Install Required Packages

Use Python 3.12 with a Jupyter kernel. The following cell installs the package versions used to verify this notebook, including its plotting and certificate dependencies. Internet access is required for installation and the initial data download. If packages were already imported before installation, restart the kernel after installation and continue from the imports cell.


In [ ]:
import sys
import subprocess

print(sys.executable)
requirements = [
    "gym-trading-env==0.3.5",
    "gymnasium==1.2.1",
    "numpy==2.2.6",
    "pandas==2.3.3",
    "matplotlib==3.10.7",
    "certifi==2025.10.5",
    "pyecharts==2.0.9",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", *requirements])


In [1]:
import numpy as np
import pandas as pd
import gymnasium as gym
import gym_trading_env
import json
import ssl
import certifi
from urllib.request import urlopen
from urllib.parse import urlencode
from pathlib import Path
import matplotlib.pyplot as plt
import time



### Prepare Data Set

- Create a data folder to store historical data
- Download historical data for BTC/USDT from Binance using the public Binance market-data API (no API key required).
- Preprocess the data to create features. The features (plus two dynamic features: last position taken by the agent, and the current real position) are the state of the environment at each time step.  
    *(You can add more features if you want to experiment with different state representations.)*
- Select training and evaluation data based on the specified date ranges.  
    *(You can modify the training range if you want to experiment with different time periods, however, keep in mind the evaluation period should always be after the training period.)*

All timestamps are UTC. Ranges are start-inclusive and end-exclusive. Download seven extra days before training for feature warm-up. Validate every expected hourly candle before using or caching the data; missing data raises an error instead of silently shortening evaluation. A dated cache keeps older datasets intact.

Source: [Binance Kline API](https://developers.binance.com/docs/binance-spot-api-docs/rest-api/market-data-endpoints#klinecandlestick-data).


In [ ]:
# Support running from either the Project folder or the repository root.
project_folder = Path("Project") if Path("Project/Project_Instruction.ipynb").is_file() else Path(".")
data_folder = project_folder / "data"
data_folder.mkdir(parents=True, exist_ok=True)
eval_folder = project_folder / "eval"
eval_folder.mkdir(parents=True, exist_ok=True)

# Fixed evaluation window shared by all students; end is exclusive.
EVAL_START = pd.Timestamp("2026-08-17", tz="UTC")
EVAL_END = pd.Timestamp("2026-09-17", tz="UTC")
TRAIN_START = EVAL_START - pd.DateOffset(years=1)
DOWNLOAD_START = TRAIN_START - pd.Timedelta(days=7)
data_path = data_folder / f"binance-BTCUSDT-1h-{DOWNLOAD_START:%Y%m%d}-{EVAL_END:%Y%m%d}.pkl"


def validate_hourly(frame, start, end, label):
    expected = pd.date_range(start, end, freq="h", inclusive="left")
    missing = expected.difference(frame.index)
    unexpected = frame.index.difference(expected)
    if (frame.empty or frame.index.has_duplicates or not frame.index.is_monotonic_increasing
            or len(missing) or len(unexpected)):
        raise ValueError(f"{label}: incomplete/invalid hourly data; "
                         f"missing={len(missing)}, unexpected={len(unexpected)}, "
                         f"first missing={missing[:5].tolist()}")
    values = frame[["open", "high", "low", "close", "volume"]]
    if not np.isfinite(values.to_numpy()).all():
        raise ValueError(f"{label}: non-finite OHLCV values")
    if ((values[["open", "high", "low", "close"]] <= 0).any().any()
            or (values["volume"] < 0).any()
            or (values["high"] < values[["open", "low", "close"]].max(axis=1)).any()
            or (values["low"] > values[["open", "high", "close"]].min(axis=1)).any()):
        raise ValueError(f"{label}: invalid OHLCV values")
    print(f"{label}: {len(frame):,} candles, {frame.index.min()} to {frame.index.max()}; no missing hours")


if data_path.exists():
    df = pd.read_pickle(data_path)
else:
    rows = []
    cursor = int(DOWNLOAD_START.timestamp() * 1000)
    end_ms = int(EVAL_END.timestamp() * 1000)
    while cursor < end_ms:
        query = urlencode(dict(symbol="BTCUSDT", interval="1h", startTime=cursor,
                               endTime=end_ms - 1, limit=1000))
        with urlopen("https://data-api.binance.vision/api/v3/klines?" + query, timeout=30,
                     context=ssl.create_default_context(cafile=certifi.where())) as response:
            batch = json.load(response)
        if not isinstance(batch, list) or not batch:
            raise RuntimeError(f"Binance returned no usable data at {cursor}: {batch}")
        next_cursor = int(batch[-1][0]) + 3_600_000
        if next_cursor <= cursor:
            raise RuntimeError("Binance pagination did not advance")
        rows.extend(batch)
        cursor = next_cursor
        time.sleep(0.1)
    df = pd.DataFrame([row[1:6] for row in rows],
                      columns=["open", "high", "low", "close", "volume"], dtype=float,
                      index=pd.to_datetime([row[0] for row in rows], unit="ms", utc=True))
    df.index.name = "date_open"
    df["date_close"] = pd.to_datetime([row[6] for row in rows], unit="ms", utc=True)

validate_hourly(df, DOWNLOAD_START, EVAL_END, "Downloaded data")
df.to_pickle(data_path)

# Causal features: all rolling windows use only present/past candles.
df["feature_close"] = df["close"].pct_change()
df["feature_open"] = df["open"] / df["close"]
df["feature_high"] = df["high"] / df["close"]
df["feature_low"] = df["low"] / df["close"]
df["feature_volume"] = df["volume"] / df["volume"].rolling(7 * 24).max()
df = df.replace([np.inf, -np.inf], np.nan).dropna()

df_train = df.loc[(df.index >= TRAIN_START) & (df.index < EVAL_START)].copy()
df_eval = df.loc[(df.index >= EVAL_START) & (df.index < EVAL_END)].copy()
validate_hourly(df_train, TRAIN_START, EVAL_START, "Training")
validate_hourly(df_eval, EVAL_START, EVAL_END, "Evaluation")
assert df_train.index.max() < df_eval.index.min(), "Train/eval overlap"
print(f"Validated data saved to {data_path}")


In [ ]:
df_train

In [ ]:
df_eval

### Setting up the Trading Environment

We use the `df_train` DataFrame for training and `df_eval` DataFrame for evaluation.

The `positions` parameter defines the discrete positions the agent can take, it is a list containing possible position values. A position value corresponds to the ratio of the portfolio valuation engaged in the position ( > 0 to bet on the rise, < 0 to bet on the decrease)

- if `position < 0` : the agent is shorting the asset
- if `position = 0` : the agent is out of the market
- if `position > 0` : the agent is longing the asset
- if `position = 1` : the agent is fully invested in the asset
- if `position > 1` : the agent is using leverage to invest more than its portfolio valuation in the asset

You are free to modify the `positions` list to experiment with different position options for the agent.


In [ ]:
def prepare_env_data(frame):
    # TradingEnv stores history dates without timezone information.
    # Keep UTC clock times while matching that format for save_for_render().
    frame = frame.copy()
    frame.index = pd.to_datetime(frame.index, utc=True).tz_localize(None)
    if "date_close" in frame.columns:
        frame["date_close"] = pd.to_datetime(frame["date_close"], utc=True).dt.tz_localize(None)
    return frame


POSITIONS = [ -1, 0, 0.5, 1, 2] # -1 (=SHORT), 0(=OUT), +1 (=LONG)

env_train = gym.make("TradingEnv",
        name= "BTCUSD",
        df = prepare_env_data(df_train), # Your dataset with your custom features
        positions = POSITIONS,
        trading_fees = 0.01/100, # 0.01% per stock buy / sell (Binance fees)
        borrow_interest_rate= 0.0003/100, # 0.0003% per timestep (one timestep = 1h here)
    )

env_eval = gym.make("TradingEnv",
        name= "BTCUSD",
        df = prepare_env_data(df_eval), # Your dataset with your custom features
        positions = POSITIONS,
        trading_fees = 0.01/100, # 0.01% per stock buy / sell (Binance fees)
        borrow_interest_rate= 0.0003/100, # 0.0003% per timestep (one timestep = 1h here)
    )

### Example of interacting with the Environment

The interaction with the environment follows the standard Gymnasium API. At each time step, the agent selects an action (position index) based on the current observation, and the environment returns the next observation, reward, and done flag.

The following code block demonstrates a simple interaction loop where the agent randomly selects actions.

The episode is terminated when `done` or `truncated` is `True`:

- When environment reaches the end of the dataset, `truncated` is set to `True`.
- When agent's portfolio valuation drops below 0, `done` is set to `True`. (This means the agent has gone bankrupt, the situdation can happen when high leverage is used.)

The reward at each time step is calculated based on the change in portfolio valuation, taking into account trading fees and borrow interest rates: $r_{t} = ln(\frac{p_{t}}{p_{t-1}})\text{ with }p_{t}\text{ = portofolio valuation at timestep }t$. You can customize your own reward function if needed (see https://gym-trading-env.readthedocs.io/en/latest/customization.html#custom-reward-function)

In [ ]:
done, truncated = False, False
observation, info = env_train.reset()
while not done and not truncated:
    # Pick a position by its index in your position list
    position_index = env_train.action_space.sample()
    observation, reward, done, truncated, info = env_train.step(position_index)

### DRL Agent Training and Evaluation

First implement a DRL algorithm covered in the course and train an initial agent using the training environment. Analyse the results, then optimise your agent through motivated experiments and comparisons with this baseline. Use the provided evaluation function to generate the final evaluation CSV for submission.

Your agent should include a method `choose_action_eval` for selecting actions based on state during evaluation. It is called in the evaluation function.

```python
    def choose_action_eval(self, state):
        # Implement action selection logic for evaluation
        ...
        return action_index
```

After training, you can evaluate your agent using the provided `evaluate_agent` function, which runs the agent in the evaluation environment for a specified number of episodes and records the results.

In [ ]:
class RandomAgent:
    def __init__(self, action_space):
        self.action_space = action_space

    def act(self, observation):
        return self.action_space.sample()

    def choose_action_eval(self, state):
        return self.act(state)


In [ ]:
def evaluate_agent(agent, env, num_episodes=10, max_steps=None, render=False, csv_path="evaluation_results.csv", renderer_logs_dir="render_logs"):
    """
    Evaluate the agent on the environment for a number of episodes.
    """

    results = []
    for ep in range(num_episodes):
        obs, info = env.reset()
        done = False
        truncated = False
        step = 0
        reward_total = 0.0
        while not done and not truncated:
            action = agent.choose_action_eval(obs)
            obs, reward, done, truncated, info = env.step(action)
            reward_total += reward
            step += 1
            if (max_steps is not None) and (step >= max_steps):
                break

        # Get metrics from the environment
        metrics = env.get_metrics()  
        # Transform metrics
        # Assume metrics contain keys "Portfolio Return" and "Market Return" as strings like "45.24%"
        port_ret = float(metrics["Portfolio Return"].strip('%')) / 100.0
        market_ret = float(metrics["Market Return"].strip('%')) / 100.0

        results.append({
            "episode": ep+1,
            "portfolio_return": port_ret,
            "market_return": market_ret,
            "excess_return": port_ret - market_ret,
            "steps": step,
            "total_reward": reward_total,
        })
        if render:
            print(f"Eval Episode {ep+1}: Total Reward: {reward_total:.2f}, Portfolio Return: {port_ret:.2%}, Market Return: {market_ret:.2%}, Excess Return: {(port_ret - market_ret):.2%}, Steps: {step}")
            time.sleep(1)  # Pause between episodes in case the execution is too fast and files are not saved properly
            env.save_for_render(dir = renderer_logs_dir)

    df_results = pd.DataFrame(results)
    
    df_results.to_csv(csv_path, index=False)
    print(f"Saved submission to {csv_path}")

    return df_results


In [ ]:
# Create a random agent for evaluation
agent = RandomAgent(env_eval.action_space)

# Evaluate the trained agent
df_results = evaluate_agent(agent, env_eval, num_episodes=10, render=True, csv_path=eval_folder / "evaluation_results.csv", renderer_logs_dir=eval_folder / "render_logs")

### Environment rendering

Gym Trading Env supports rendering the environment to visualize the agent's trading actions over time. You can enable rendering during evaluation by setting the `render` parameter to `True` in the `evaluate_agent` function. The rendered logs will be saved in the specified directory for later review.

To visualize the rendered logs, you can use the built-in rendering tools provided by Gym Trading Env as shown below.

In [ ]:
from gym_trading_env.renderer import Renderer
renderer = Renderer(render_logs_dir=eval_folder/"render_logs")
renderer.run()

Now you are ready to implement your DRL agent and start training! Good luck 💪